# Combining Datasets with Pandas

This notebook teaches the main ways to combine datasets using **3 files from the Datos Wansoft dataset**.

Each file represents sales transactions from a different Panem restaurant branch:
- `Carreta` — Feb–May 2022
- `Punto Valle` — Feb–May 2022
- `Plaza QIN` — Apr–May 2022

---

## Methods covered

| Method | Use case |
|---|---|
| `pd.concat()` axis=0 | Stack rows (same columns, different data) |
| `pd.concat()` axis=1 | Stack columns side by side |
| `glob` + loop | Load and combine many files at once |

---
## Example: Load the 3 files

In [2]:

import pandas as pd
import glob
import os
import numpy
DATA_DIR = "Complete Data/"

# --- Load the 3 files individually ---
df_carreta = pd.read_csv(DATA_DIR + "detail_Panem-Carreta_2022-02-18_2022-05-07.csv")
df_punto   = pd.read_csv(DATA_DIR + "detail_Panem-Punto-Valle_2022-02-08_2022-05-08.csv")
df_qin     = pd.read_csv(DATA_DIR + "detail_Panem-Plaza-QIN_2022-04-28_2022-05-08.csv")

print("Shapes:")
print(f"  Carreta:     {df_carreta.shape}")
print(f"  Punto Valle: {df_punto.shape}")
print(f"  Plaza QIN:   {df_qin.shape}")

# Quick look at one of them
df_carreta.head(3)

Shapes:
  Carreta:     (21524, 46)
  Punto Valle: (51068, 46)
  Plaza QIN:   (5067, 46)


,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
0,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:48.077000,7,1,1,Para llevar,'-,...,201.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DE005
1,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:52.763000,7,1,1,Para llevar,'-,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA001
2,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:53.950000,7,1,1,Para llevar,'-,...,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA003


---
## `pd.concat()` axis=0: Stack rows vertically

**When to use:** All files have the **same columns** and you want to build one unified table.

This is the most common operation when files represent different time periods or locations with the same schema.

In [3]:
# --- Basic concat: just stack the rows ---
df_all = pd.concat([df_carreta, df_punto, df_qin])

print(f"Individual sizes: {len(df_carreta)}, {len(df_punto)}, {len(df_qin)}")
print(f"Combined size:    {len(df_all)}  (should equal the sum above)")

# Notice the index is duplicated — rows from each file keep their original index
print("\nFirst few index values:", df_all.index[:6].tolist())
df_all.head(3)

Individual sizes: 21524, 51068, 5067
Combined size:    77659  (should equal the sum above)

First few index values: [0, 1, 2, 3, 4, 5]


,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
0,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:48.077000,7,1,1,Para llevar,'-,...,201.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DE005
1,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:52.763000,7,1,1,Para llevar,'-,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA001
2,Panem - Carreta,2022-02-18,viernes,2022-02-18 08:21:04.887000,2022-02-18 08:17:53.950000,7,1,1,Para llevar,'-,...,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MA003


In [4]:
# --- ignore_index=True resets the index so it is clean and continuous ---
df_all = pd.concat([df_carreta, df_punto, df_qin], ignore_index=True)

print("Index after ignore_index=True:", df_all.index[:6].tolist())
print(f"Shape: {df_all.shape}")

Index after ignore_index=True: [0, 1, 2, 3, 4, 5]
Shape: (77659, 46)


---
## `glob` + loop: Combine many files at once

**When to use:** You have dozens of files with the same schema and don't want to load them one by one.

This is the real-world pattern for this dataset (150+ files).

In [5]:
# --- Load every Carreta file (all periods) with one pattern ---
carreta_files = glob.glob(DATA_DIR + "detail_Panem-Carreta_*.csv")
print(f"Carreta files found: {len(carreta_files)}")
for f in sorted(carreta_files):
    print(" ", f.split("/")[-1])

Carreta files found: 7
  detail_Panem-Carreta_2022-02-18_2022-05-07.csv
  detail_Panem-Carreta_2022-05-09_2022-08-05.csv
  detail_Panem-Carreta_2022-08-08_2022-11-04.csv
  detail_Panem-Carreta_2022-11-05_2023-02-02.csv
  detail_Panem-Carreta_2023-02-03_2023-05-03.csv
  detail_Panem-Carreta_2023-05-04_2023-08-01.csv
  detail_Panem-Carreta_2023-08-02_2023-10-30.csv


In [6]:
# --- Combine all of them into one DataFrame ---
df_carreta_full = pd.concat(
    [pd.read_csv(f) for f in sorted(carreta_files)],
    ignore_index=True
)

print(f"Total Carreta rows (all periods): {len(df_carreta_full):,}")
print(f"Date range: {df_carreta_full['operating_date'].min()}  →  {df_carreta_full['operating_date'].max()}")

/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304/4152468081.py:3: DtypeWarning: Columns (0: is_modifier) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in sorted(carreta_files)],


Total Carreta rows (all periods): 171,360
Date range: 2022-02-18  →  2023-10-30


In [7]:
# --- Or load ALL files from ALL branches at once ---
all_files = glob.glob(DATA_DIR + "detail_*.csv")
print(f"Total files: {len(all_files)}")

df_complete = pd.concat(
    [pd.read_csv(f) for f in all_files],
    ignore_index=True
)

print(f"Total rows: {len(df_complete):,}")
print("\nRows per branch:")
print(df_complete["sucursal"].value_counts())

Total files: 186


/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304/2627673863.py:6: DtypeWarning: Columns (0: is_modifier) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304/2627673863.py:6: DtypeWarning: Columns (0: is_modifier) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304/2627673863.py:6: DtypeWarning: Columns (0: is_modifier) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304/2627673863.py:6: DtypeWarning: Columns (0: is_modifier) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in all_files],
/var/folders/x6/jnp28xtj0r3fy6zsyhywxpxc0000gn/T/ipykernel_30304

Total rows: 3,764,683

Rows per branch:
sucursal
Panem - Punto Valle            944346
Panem - Hotel Kavia N          602608
Panem - Plaza QIN N            385516
Panem - Plaza QIN              349364
Panem - Hospital Zambrano N    335642
Panem - Hotel Kavia            254154
Panem - Hospital Zambrano      251304
Panem - La Carreta N           227499
Panem - Plaza Nativa           194511
Panem - Carreta                171360
Panem - Credi Club              48379
Name: count, dtype: int64


In [8]:
df_complete.describe()

,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,...,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
count,3764683,3764683,3764683,3764683,3764683,3764683,3764683,3764683,3764683,3764683,...,3753257.0,39361.00,39361.0,39361.0,39361.0,10692.0,10692.0,10692.0,10692.0,3764683
unique,11,1473,7,1029804,3492948,52,234433,370,2,1,...,2780.0,564.00,456.0,26.0,330.0,358.0,272.0,52.0,288.0,821
top,Panem - Punto Valle,2026-02-07,sábado,2023-09-20 15:42:31.757000,2023-07-04 10:45:00,48,66579,22,Restaurant,'-,...,0.0,63.79,0.0,0.0,74.0,0.0,0.0,0.0,0.0,MB006
freq,944346,9894,635486,123,16,96395,140,28508,2075655,3764683,...,1081804.0,4685.00,6242.0,39179.0,5504.0,1386.0,3207.0,10024.0,1386.0,248171


In [9]:
df_complete.shape

(3764683, 46)

In [10]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

df_complete.head(10).T

,0,1,2,3,4,5,6,7,8,9
sucursal,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N,Panem - Plaza QIN N
operating_date,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22,2025-12-22
day_name,lunes,lunes,lunes,lunes,lunes,lunes,lunes,lunes,lunes,lunes
closing_time,2025-12-22 08:04:56.373000,2025-12-22 08:04:56.373000,2025-12-22 08:05:58.817000,2025-12-22 08:05:58.817000,2025-12-22 08:06:42.833000,2025-12-22 08:14:30.573000,2025-12-22 08:14:30.573000,2025-12-22 08:26:32.450000,2025-12-22 08:26:32.450000,2025-12-22 08:26:32.450000
captured_time,2025-12-22 08:03:54.367000,2025-12-22 08:04:00.267000,2025-12-22 08:05:06.900000,2025-12-22 08:05:08.707000,2025-12-22 08:06:10.007000,2025-12-22 08:14:03.263000,2025-12-22 08:14:03.980000,2025-12-22 08:23:41.357000,2025-12-22 08:24:52.360000,2025-12-22 08:24:53.280000
week_number,52,52,52,52,52,52,52,52,52,52
pdv_txn_id,219634,219634,219635,219635,219636,219637,219637,219638,219638,219638
order_id,1,1,2,2,3,4,4,5,5,5
order_type,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar,Para llevar
order_subtype,'-,'-,'-,'-,'-,'-,'-,'-,'-,'-


In [11]:
from numpy import dtype

df_complete.dtypes.to_frame("dtype")

,dtype
sucursal,object
operating_date,object
day_name,object
closing_time,object
captured_time,object
week_number,object
pdv_txn_id,object
order_id,object
order_type,object
order_subtype,object


In [12]:
duplicated_values = df_complete.duplicated().sum()
print(duplicated_values)

229108


In [13]:
# Show first 10 duplicated rows
dup_rows = df_complete[df_complete.duplicated()]
dup_rows.head(10)


,sucursal,operating_date,day_name,closing_time,captured_time,week_number,pdv_txn_id,order_id,order_type,order_subtype,table_number,party_size,server,terminal,capture_terminal,action,item,modifier,group_type,group,description,is_modifier,quantity,unit_price,unit_price_with_mods,cost_actual,cost_with_mods,cost_ideal,discount,subtotal_ticket,iva_ticket,ieps_ticket,total_ticket,subtotal_item,iva_item,ieps_item,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion,clave_platillo
7696,Panem - Hospital Zambrano,2022-05-24,martes,2022-05-24 18:24:21.043000,2022-05-24 18:17:00,21,1885,118,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Cancelación de platillo,COCA COLA,NaN,Alimentos,JUGOS Y BEBIDAS FRIAS,"Borrado de Partida Impresa, cambio de comida",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.2069,5.7931,0.0,42.0,NaN,NaN,NaN,NaN,JBF013
9587,Panem - Hospital Zambrano,2022-05-29,domingo,2022-05-29 18:40:12.343000,2022-05-29 18:16:00,21,2484,52,Restaurant,'-,0.0,1,Zambrano,SERVER1,NaN,Cancelación de platillo,PAIN AU CHOCOLAT,NaN,Alimentos,PAN DULCE,"Borrado de Partida Impresa, ERROR",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.7931,10.2069,0.0,74.0,NaN,NaN,NaN,NaN,PD005
17379,Panem - Hospital Zambrano,2022-06-17,viernes,2022-06-17 16:31:27.120000,2022-06-17 16:31:00,24,5007,97,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Cancelación de platillo,CHIPS LOCAL SNACKS,NaN,Alimentos,PANEM MARKETPLACE,"Borrado de Partida Impresa, NO QUIZO",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.7241,8.2759,0.0,60.0,NaN,NaN,NaN,NaN,PM015
21060,Panem - Hospital Zambrano,2022-06-27,lunes,2022-06-27 10:03:06.280000,2022-06-27 10:02:00,26,6157,8,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Cancelación de platillo,CONCHA CHOCOLATE,NaN,Alimentos,PAN DULCE,"Borrado de Partida Impresa, cambio panh",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.6897,3.3103,0.0,24.0,NaN,NaN,NaN,NaN,PD020
45153,Panem - Hotel Kavia N,2024-05-07,martes,2024-05-07 12:05:19.423000,2024-05-07 12:05:00,19,105748,23,Restaurant,'-,1.0,1,Panem Kavia,Terminal 1,NaN,Anulación de platillo,HUEVO,NaN,Alimentos,EXTRAS,"Borrado de Partida Impresa, ESC",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.3448,1.6552,0.0,12.0,MA025
49873,Panem - Hotel Kavia N,2024-05-13,lunes,2024-05-13 22:44:49.247000,2024-05-13 22:29:00,20,106642,76,Restaurant,'-,1.0,1,Panem Kavia,Terminal 1,NaN,Anulación de platillo,BAGUETTE PETITE,NaN,Alimentos,PAN SALADO,"Borrado de Partida Impresa, prueba",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.2759,3.7241,0.0,27.0,PS005
115598,Panem - Hospital Zambrano,2023-05-31,miércoles,2023-05-31 11:51:46.607000,2023-05-31 11:45:00,22,53811,54,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Cancelación de platillo,CONCHA VAINILLA,NaN,Alimentos,PAN DULCE,"Borrado de Partida Impresa, se cambio a otra cuenta",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.6897,3.3103,0.0,24.0,NaN,NaN,NaN,NaN,PD001
115600,Panem - Hospital Zambrano,2023-05-31,miércoles,2023-05-31 11:51:46.607000,2023-05-31 11:45:00,22,53811,54,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Cancelación de platillo,ESPRESSO SIMPLE,NaN,Alimentos,CAFE Y BEBIDAS CALIENTES,"Borrado de Partida Impresa, se cambio a otra cuenta",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.7931,6.2069,0.0,45.0,NaN,NaN,NaN,NaN,CBC008
131610,Panem - Hospital Zambrano,2023-06-30,viernes,2023-06-30 15:07:43.937000,2023-06-30 15:06:00,26,58625,93,Para llevar,'-,NaN,1,Zambrano,SERVER1,NaN,Anulación de platillo,AGUA,NaN,Alimentos,JUGOS Y BEBIDAS FRIAS,"Borrado de Partida Impresa, error",NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.6207,5.3793,0.0,39.0,JBF014
133371,Panem - Hospital Zambrano,2023-07-04,martes,2023-07-04 10:22:43.077000,20

In [14]:
df_complete.shape, df_complete.isna().sum()

((3764683, 46),
 sucursal                          0
 operating_date                    0
 day_name                          0
 closing_time                      0
 captured_time                     0
 week_number                       0
 pdv_txn_id                        0
 order_id                          0
 order_type                        0
 order_subtype                     0
 table_number                1689028
 party_size                        0
 server                            0
 terminal                          0
 capture_terminal               6140
 action                            0
 item                              0
 modifier                    2579572
 group_type                        0
 group                             0
 description                       0
 is_modifier                    6140
 quantity                          0
 unit_price                     6140
 unit_price_with_mods           6140
 cost_actual                    6140
 cost_with_mods       

In [15]:
#Eliminamos valores no existentes y cambiamos el tipo de dato a string
df_complete["modifier"] = df_complete["modifier"].fillna("No modificacion").astype("string")

In [16]:
#Eliminamos los valores nulos y convertimos la columna a entero
df_complete["table_number"] = df_complete["table_number"].fillna(0).astype("int")

In [17]:
#Eliminamos los valores nulos y convertimos la columna a entero
df_complete["capture_terminal"] = df_complete["capture_terminal"].fillna("NULL").astype("str")

In [18]:
#Definir columnas

columnas_int = ["party_size","week_number", "pdv_txn_id", "order_id", "table_number"]
columnas_float = ["quantity", "unit_price", "unit_price_with_mods", "cost_actual", "cost_with_mods", "cost_ideal", "discount", "subtotal_ticket", "iva_ticket", "ieps_ticket", "total_ticket", "subtotal_item", "iva_item", "ieps_item", "total_item", "subtotal_cortesia_cancel", "iva_cortesia_cancel", "ieps_cortesia_cancel", "total_cortesia_cancel", "subtotal_anulacion", "iva_anulacion", "ieps_anulacion", "total_anulacion"]
columnas_datetime = ["operating_date", "closing_time", "captured_time"]
columnas_str = ["sucursal", "day_name", "order_type", "order_subtype", "server", "terminal", "capture_terminal", "action", "item", "modifier", "group_type", "group", "description", "clave_platillo"]
columnas_bool = ["is_modifier"]



In [19]:
for column in columnas_float:
    df_complete[columnas_float] = df_complete[columnas_float].fillna(0).astype("float64")


In [20]:

for column in columnas_datetime:
    if column in df_complete.columns:
        df_complete[column] = pd.to_datetime(df_complete[column], errors='coerce')

In [21]:
for column in columnas_int:
  df_complete[column] = df_complete[column].astype("int")


In [22]:
for column in columnas_str:
    df_complete[column] = df_complete[column].fillna("N/A").astype("string")

In [23]:

for column in columnas_bool:
    df_complete[column] = df_complete[column].astype("bool")

In [24]:
df_complete.shape

(3764683, 46)

In [25]:
df_complete.shape, df_complete.isna().sum()

((3764683, 46),
 sucursal                        0
 operating_date                  0
 day_name                        0
 closing_time                12504
 captured_time               18694
 week_number                     0
 pdv_txn_id                      0
 order_id                        0
 order_type                      0
 order_subtype                   0
 table_number                    0
 party_size                      0
 server                          0
 terminal                        0
 capture_terminal                0
 action                          0
 item                            0
 modifier                        0
 group_type                      0
 group                           0
 description                     0
 is_modifier                     0
 quantity                        0
 unit_price                      0
 unit_price_with_mods            0
 cost_actual                     0
 cost_with_mods                  0
 cost_ideal                      0
 dis

In [26]:
df_complete.dtypes.to_frame("dtype")

,dtype
sucursal,string
operating_date,datetime64[us]
day_name,string
closing_time,datetime64[us]
captured_time,datetime64[us]
week_number,int64
pdv_txn_id,int64
order_id,int64
order_type,string
order_subtype,string


In [27]:
df_complete.describe()

,operating_date,closing_time,captured_time,week_number,pdv_txn_id,order_id,table_number,party_size,quantity,unit_price,unit_price_with_mods,cost_actual,cost_with_mods,cost_ideal,discount,subtotal_ticket,iva_ticket,ieps_ticket,total_ticket,subtotal_item,iva_item,ieps_item,total_item,subtotal_cortesia_cancel,iva_cortesia_cancel,ieps_cortesia_cancel,total_cortesia_cancel,subtotal_anulacion,iva_anulacion,ieps_anulacion,total_anulacion
count,3764683,3752179,3745989,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3764683.0,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06,3.764683e+06
mean,2024-06-03 20:09:23.475651,2024-06-04 09:41:43.208407,2024-06-04 12:53:49.523593,2.683446e+01,1.126959e+05,8.321795e+01,8.026907e+00,1.342820e+00,1.099272e+00,5.656830e+01,5.656830e+01,4.820697e+00,4.820928e+00,1.326798e+01,0.0,2.721145e+02,3.878044e+01,2.180312e+00,3.130753e+02,5.147635e+01,7.160861e+00,4.819341e-01,5.911915e+01,8.621355e-01,1.205533e-01,2.683126e-04,9.910442e-01,2.082496e-01,2.818185e-02,9.786890e-04,2.386726e-01
min,2022-01-31 00:00:00,2022-01-31 20:27:20.527000,2022-01-31 20:25:46.030000,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,6.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2023-07-06 00:00:00,2023-07-06 14:30:02.830000,2023-07-06 17:53:00.137000,1.300000e+01,6.151100e+04,3.500000e+01,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,1.017242e+02,1.241380e+01,0.000000e+00,1.160000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2024-07-06 00:00:00,2024-07-06 17:04:16.283000,2024-07-06 18:50:36.523000,2.700000e+01,1.138360e+05,7.300000e+01,1.000000e+00,1.000000e+00,1.000000e+00,4.500000e+01,4.900000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.0,2.077586e+02,2.813790e+01,0.000000e+00,2.400000e+02,4.224140e+01,5.379300e+00,0.000000e+00,4.900000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2025-05-25 00:00:00,2025-05-25 10:10:09.887000,2025-05-25 11:17:25.860000,4.100000e+01,1.633510e+05,1.210000e+02,1.500000e+01,1.000000e+00,1.000000e+00,8.500000e+01,8.500000e+01,1.347000e+00,1.311000e+00,7.320000e+00,0.0,3.672414e+02,5.241380e+01,1.925900e+00,4.210000e+02,7.672410e+01,1.158620e+01,0.000000e+00,8.900000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2026-02-12 00:00:00,2026-02-12 21:45:55.440000,2026-02-12 21:45:31.527000,5.200000e+01,2.344440e+05,3.700000e+02,9.999000e+03,9.999000e+03,1.100000e+03,1.000000e+03,1.000000e+03,4.875915e+03,4.875915e+03,3.289000e+04,0.0,2.198276e+04,3.517241e+03,5.365926e+02,2.550000e+04,1.100000e+04,1.464828e+03,2.222222e+02,1.100000e+04,1.241379e+04,1.986210e+03,2.555560e+01,1.440000e+04,4.362069e+03,6.979310e+02,5.777780e+01,5.060000e+03
std,NaN,NaN,NaN,1.545938e+01,6.274208e+04,5.952749e+01,5.310385e+01,3.836051e+01,1.774978e+00,5.832164e+01,6.008611e+01,1.847796e+01,1.848818e+01,6.936820e+01,0.0,2.730931e+02,4.238279e+01,5.384488e+00,3.152874e+02,6.075064e+01,9.660633e+00,1.722561e+00,6.986322e+01,1.526884e+01,2.322132e+00,4.301749e-02,1.758279e+01,6.502484e+00,9.978156e-01,9.506999e-02,7.496808e+00


In [28]:
df_complete.describe(include=["object", "string"]).T


,count,unique,top,freq
sucursal,3764683,11,Panem - Punto Valle,944346
day_name,3764683,7,sábado,635486
order_type,3764683,2,Restaurant,2075655
order_subtype,3764683,1,'-,3764683
server,3764683,36,Pruebas,944338
terminal,3764683,9,SERVER1,2000023
capture_terminal,3764683,10,SERVER1,1995999
action,3764683,7,Venta,3714630
item,3764683,666,CAFE REFILL,425201
modifier,3764683,209,No modificacion,2579572


In [29]:
cat_cols = df_complete.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df_complete[col].value_counts(dropna=False).head(5))




--- sucursal ---
sucursal
Panem - Punto Valle            944346
Panem - Hotel Kavia N          602608
Panem - Plaza QIN N            385516
Panem - Plaza QIN              349364
Panem - Hospital Zambrano N    335642
Name: count, dtype: int64[pyarrow]

--- day_name ---
day_name
sábado       635486
viernes      560893
domingo      554078
jueves       526999
miércoles    510377
Name: count, dtype: int64[pyarrow]

--- order_type ---
order_type
Restaurant     2075655
Para llevar    1689028
Name: count, dtype: int64[pyarrow]

--- order_subtype ---
order_subtype
'-    3764683
Name: count, dtype: int64[pyarrow]

--- server ---
server
Pruebas        944338
Panem Kavia    808024
Plaza Qin      734747
Zambrano       586360
Paseo Tec      194511
Name: count, dtype: int64[pyarrow]

--- terminal ---
terminal
SERVER1       2000023
Terminal 1    1214803
server_qin     349364
PDV1           194563
PDV2             5110
Name: count, dtype: int64[pyarrow]

--- capture_terminal ---
capture_terminal
SERVE

In [30]:
df_complete["group"].value_counts().head(50)

group
MOD BEBIDAS                       861397
PAN DULCE                         683045
CAFE Y BEBIDAS CALIENTES          638602
JUGOS Y BEBIDAS FRIAS             530067
DESAYUNOS                         332655
MOD ALIMENTOS                     199587
COMIDAS                           160861
REPOSTERIA                        129362
EXTRAS                             50000
PIZZA                              38674
PRODUCTOS DE TEMPORADA             27744
NATIVA TEMPORALCOMIDA              21417
PAN SALADO                         15384
UBER PAN DULCE                     11644
PANEM MARKETPLACE                  10897
SUBSIDIO                            9134
RAPPI PAN DULCE                     5904
MOD MARKETPLACE                     5485
UBER DESAYUNOS                      5113
UBER COMIDAS                        4452
ESPECIALES-                         4387
UBER CAFE Y BEBIDAS CALIENTES       4202
UBER JUGOS Y BEBIDAS FRIAS          2039
UBER REPOSTERIA                     1934
RAPPI DESA

In [31]:
df_complete.select_dtypes(include="number").agg(["min", "max"]).T


,min,max
week_number,1.0,52.0000
pdv_txn_id,1.0,234444.0000
order_id,1.0,370.0000
table_number,0.0,9999.0000
party_size,1.0,9999.0000
quantity,0.6,1100.0000
unit_price,0.0,1000.0000
unit_price_with_mods,0.0,1000.0000
cost_actual,0.0,4875.9154
cost_with_mods,0.0,4875.9154


In [32]:
impossible_party_size = (df_complete["party_size"] > 15).sum()
print(impossible_party_size)


3736


In [33]:
df_complete["server"].value_counts(dropna=False)



server
Pruebas                                   944338
Panem Kavia                               808024
Plaza Qin                                 734747
Zambrano                                  586360
Paseo Tec                                 194511
BRENDA YAHAIRA MARTINEZ HERNDANDEZ        180423
Pruebas Carreta                           171360
PRUEBAS                                    48379
Jose                                       47068
Fabiola Sarahi Quezada Hernández           21688
Aaron Abdiel Ramirez Mancisidor            19605
Alan Octavio Hipolito Escareño              3936
CYNTHIA DEYANIRA ACOSTA ARREDONDO           1565
Estefania Guadalupe Arambula Ontiveros       601
SAUL ADRIAN MARTINEZ MIRELES                 540
MARTINEZ SAAVEDRA                            486
RAMIRO ISLA QUIROZ                           265
AMY YARELI TORRES CUELLAR                    176
ALEXIA MONSERRAT FLORES ACOSTA               123
MIRIAM JANET GODINA RAMIREZ                   95
MISAHEL ANTON

In [34]:
# Only rows where server is Pruebas and quantity > 100
filtro = (df_complete["server"] == "Pruebas") & (df_complete["quantity"] > 0)

# Count repetitions of sucursal
df_complete.loc[filtro, "sucursal"].value_counts(dropna=False)


sucursal
Panem - Punto Valle    944338
Name: count, dtype: int64[pyarrow]

In [35]:
output_csv = "/Users/mak/Desktop/Python challenge/df_complete_186_files.csv"
df_complete.to_csv(output_csv, index=False, encoding="utf-8")

In [36]:
df_complete.dtypes


sucursal                            string
operating_date              datetime64[us]
day_name                            string
closing_time                datetime64[us]
captured_time               datetime64[us]
week_number                          int64
pdv_txn_id                           int64
order_id                             int64
order_type                          string
order_subtype                       string
table_number                         int64
party_size                           int64
server                              string
terminal                            string
capture_terminal                    string
action                              string
item                                string
modifier                            string
group_type                          string
group                               string
description                         string
is_modifier                           bool
quantity                           float64
unit_price 